# Privacy & Governance Assessment — NovaCred Credit Application Data

**Notebook:** 03 — Privacy Demonstration & Governance Analysis  
**Author:** Martin Schmitz, Governance Officer  
**Course:** Data Ecosystems and Governance in Organizations (DEGO 2606) — Nova SBE  
**Project:** Credit Application Governance Analysis — Team 7  

---

## Purpose

This notebook conducts a **Privacy and Governance Assessment** of NovaCred's raw credit application dataset. As the team's Governance Officer, the analysis covers four areas:

1. **PII Identification** — Systematically cataloguing all personally identifiable information present in the dataset
2. **Pseudonymisation Demonstration** — Applying privacy-preserving transformations to sensitive fields
3. **GDPR Compliance Mapping** — Mapping findings to specific GDPR articles and identifying compliance gaps
4. **Governance Recommendations** — Proposing concrete controls to prevent future issues

The assessment is grounded in two regulatory frameworks:
- **GDPR** (EU 2016/679) — the primary data protection regulation governing how personal data must be collected, stored, and processed
- **EU AI Act** (2024/1689) — which classifies credit scoring systems as **high-risk AI**, imposing additional transparency and human oversight obligations

---

## Dataset Context

The dataset analysed in this notebook is the **cleaned output** produced by the Data Engineering pipeline (`01_data_quality.ipynb`). It contains **500 credit applications** with 34 variables spanning applicant identity, financial profile, spending behaviour, and loan decisions.

The **raw dataset** (`raw_credit_applications.json`) is also referenced where relevant to highlight issues present before cleaning.

---

## Section 1 — Setup

In [ ]:
import pandas as pd
import numpy as np
import hashlib
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned dataset
df = pd.read_csv('../data/processed/clean_credit_applications.csv')

print(f'Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

---

## Section 2 — PII Identification

### 2.1 Overview

Under **GDPR Article 4(1)**, personal data is defined as:

> *"any information relating to an identified or identifiable natural person"*

This covers not only obvious identifiers such as names and national ID numbers, but also any information that could be combined with other data to re-identify an individual (quasi-identifiers).

PII is classified here into three tiers:

| Tier | Definition | Examples |
|---|---|---|
| **Direct Identifier** | Uniquely identifies a person on its own | Name, SSN, email |
| **Quasi-Identifier** | Can re-identify when combined with other fields | ZIP code, age, gender |
| **Sensitive Behavioral Data** | Reveals personal habits or lifestyle; additional protection warranted | Gambling, alcohol, adult entertainment spend |

The dataset is examined systematically for all three tiers below.

In [ ]:
# Define PII inventory
pii_inventory = [
    {
        'Field': 'ssn',
        'PII Classification': 'Direct Identifier',
        'Risk Level': 'Critical',
        'GDPR Relevance': 'National ID number — Art. 4(1); stored in plaintext, no encryption or pseudonymisation applied',
        'Recommended Action': 'Pseudonymise immediately (SHA-256 hash or tokenise); restrict access to authorised personnel only'
    },
    {
        'Field': 'full_name',
        'PII Classification': 'Direct Identifier',
        'Risk Level': 'Critical',
        'GDPR Relevance': 'Direct personal identifier — Art. 4(1); not needed after identity verification stage',
        'Recommended Action': 'Pseudonymise or remove post-onboarding; retain only in identity-verified system of record'
    },
    {
        'Field': 'email',
        'PII Classification': 'Direct Identifier',
        'Risk Level': 'Critical',
        'GDPR Relevance': 'Contact data — Art. 4(1); links decisions to identifiable person; must have documented lawful basis',
        'Recommended Action': 'Pseudonymise in analytical datasets; retain only in CRM with access controls'
    },
    {
        'Field': 'ip_address',
        'PII Classification': 'Direct Identifier',
        'Risk Level': 'High',
        'GDPR Relevance': 'Personal data under GDPR Recital 30; can identify device/user; retained beyond application processing without clear justification',
        'Recommended Action': 'Remove from analytical dataset; if retained for fraud detection, define retention period and document lawful basis'
    },
    {
        'Field': 'date_of_birth',
        'PII Classification': 'Direct Identifier',
        'Risk Level': 'High',
        'GDPR Relevance': 'Art. 4(1); combined with name/ZIP creates strong re-identification vector; full DOB likely not needed if age is derived',
        'Recommended Action': 'Replace with derived `applicant_age` for analysis; apply data minimisation (Art. 5(1)(c))'
    },
    {
        'Field': 'zip_code',
        'PII Classification': 'Quasi-Identifier',
        'Risk Level': 'High',
        'GDPR Relevance': 'Geographic quasi-identifier; combined with gender and age enables re-identification (k-anonymity risk); correlated with protected characteristics (proxy discrimination risk)',
        'Recommended Action': 'Generalise to broader region (e.g., first 3 digits) for analytical use; assess proxy discrimination risk'
    },
    {
        'Field': 'gender',
        'PII Classification': 'Quasi-Identifier',
        'Risk Level': 'High',
        'GDPR Relevance': 'Protected characteristic under GDPR; its use in credit decisions may constitute discrimination (Art. 22 automated decision-making)',
        'Recommended Action': 'Document lawful basis for collection; exclude from automated decision model; retain only for bias monitoring with appropriate access controls'
    },
    {
        'Field': 'spend_gambling',
        'PII Classification': 'Sensitive Behavioral Data',
        'Risk Level': 'High',
        'GDPR Relevance': 'Reveals personal habits linked to financial vulnerability; collection requires explicit justification under data minimisation (Art. 5(1)(c))',
        'Recommended Action': 'Assess necessity for credit decision; if not predictive, remove; if retained, document purpose and apply strict access controls'
    },
    {
        'Field': 'spend_adult_entertainment',
        'PII Classification': 'Sensitive Behavioral Data',
        'Risk Level': 'High',
        'GDPR Relevance': 'Reveals intimate lifestyle data; highly sensitive; no clear relevance to creditworthiness; data minimisation violation (Art. 5(1)(c))',
        'Recommended Action': 'Remove from dataset immediately; collection of this category is disproportionate to the stated purpose of credit assessment'
    },
    {
        'Field': 'spend_alcohol',
        'PII Classification': 'Sensitive Behavioral Data',
        'Risk Level': 'Medium',
        'GDPR Relevance': 'Behavioral data that may indicate health-related patterns; disproportionate collection without clear credit relevance (Art. 5(1)(c))',
        'Recommended Action': 'Assess whether predictive value justifies collection; if retained, aggregate into broader spending category'
    },
]

pii_df = pd.DataFrame(pii_inventory)

print('PII Inventory — NovaCred Credit Application Dataset')
print('=' * 60)
pii_df

### 2.2 PII Field Coverage

Before assessing risk, it is important to quantify how widely each PII field is populated across the 500 records. A field that is present in nearly every record poses a greater exposure risk than one with sparse coverage.

In [ ]:
pii_fields = ['ssn', 'full_name', 'email', 'ip_address', 'date_of_birth',
              'zip_code', 'gender', 'spend_gambling', 'spend_adult_entertainment', 'spend_alcohol']

coverage = []
for field in pii_fields:
    if field in df.columns:
        populated = df[field].notna().sum()
        # For spending fields, also count non-zero values
        if field.startswith('spend_'):
            nonzero = (df[field] > 0).sum()
            coverage.append({
                'Field': field,
                'Records Populated': populated,
                'Coverage (%)': round(populated / len(df) * 100, 1),
                'Records with Non-Zero Value': int(nonzero),
                'Non-Zero Coverage (%)': round(nonzero / len(df) * 100, 1)
            })
        else:
            coverage.append({
                'Field': field,
                'Records Populated': populated,
                'Coverage (%)': round(populated / len(df) * 100, 1),
                'Records with Non-Zero Value': '-',
                'Non-Zero Coverage (%)': '-'
            })

coverage_df = pd.DataFrame(coverage)
print('PII Field Coverage across 500 records')
print('=' * 60)
coverage_df

### 2.3 Critical Finding — SSNs Stored in Plaintext

Social Security Numbers (SSNs) are among the most sensitive direct identifiers in any dataset. A breach of unencrypted SSNs constitutes a **high-severity personal data breach** under GDPR Article 33, requiring notification to the supervisory authority within 72 hours.

The check below confirms that SSNs in this dataset are stored as raw, unencrypted strings in the standard `XXX-XX-XXXX` format — with no hashing, encryption, or tokenisation applied.

In [ ]:
# Check SSN format — confirms plaintext storage
ssn_pattern = re.compile(r'^\d{3}-\d{2}-\d{4}$')

ssn_series = df['ssn'].dropna().astype(str)
plaintext_ssns = ssn_series.apply(lambda x: bool(ssn_pattern.match(x))).sum()
total_ssns = ssn_series.shape[0]

print('=== SSN Plaintext Storage Audit ===')
print(f'Total records with SSN populated : {total_ssns}')
print(f'SSNs in standard plaintext format: {plaintext_ssns} ({round(plaintext_ssns/total_ssns*100, 1)}%)')
print()

# Show sample (first 5) to demonstrate they are readable
print('Sample SSN values (first 5 records):')
print(df['ssn'].dropna().head(5).to_string())
print()

# Flag as critical finding
if plaintext_ssns > 0:
    print('CRITICAL FINDING: SSNs are stored in plaintext.')
    print(f'{plaintext_ssns} records expose raw SSN values with no protection.')
    print('This constitutes a critical data governance failure.')
    print('Remediation required: pseudonymise SSNs before any further processing or sharing.')

### 2.4 Summary Visualisation — PII Risk by Field

The chart below provides a visual summary of the coverage (exposure breadth) of each PII field, coloured by risk level. Fields with high coverage and high risk represent the greatest governance priority.

In [ ]:
# Map risk levels to colours
risk_colours = {'Critical': '#d62728', 'High': '#ff7f0e', 'Medium': '#bcbd22'}

# Merge coverage data with risk level from PII inventory
risk_map = {row['Field']: row['Risk Level'] for row in pii_inventory}
coverage_df['Risk Level'] = coverage_df['Field'].map(risk_map)
coverage_df['Colour'] = coverage_df['Risk Level'].map(risk_colours)

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    coverage_df['Field'],
    coverage_df['Coverage (%)'],
    color=coverage_df['Colour'],
    edgecolor='white',
    height=0.6
)

# Annotate bars with coverage %
for bar, val in zip(bars, coverage_df['Coverage (%)']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val}%', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in risk_colours.items()]
ax.legend(handles=legend_elements, title='Risk Level', loc='lower right')

ax.set_xlabel('Coverage (% of 500 records populated)')
ax.set_title('PII Field Coverage and Risk Level — NovaCred Dataset', fontsize=13, fontweight='bold')
ax.set_xlim(0, 115)
ax.invert_yaxis()
ax.grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('../reports/pii_risk_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/pii_risk_coverage.png')

### 2.5 Section Summary — Key Findings

The PII identification audit reveals **three critical governance failures** in the NovaCred dataset:

**1. Unprotected Direct Identifiers at Scale**  
Five direct identifiers — `full_name`, `email`, `ssn`, `ip_address`, and `date_of_birth` — are present in the dataset with no pseudonymisation, encryption, or access restriction. SSNs (100% populated) are stored in standard plaintext format, making every record a high-severity breach risk.

**2. Data Minimisation Violations (GDPR Article 5(1)(c))**  
The dataset contains granular spending categories for `gambling`, `adult_entertainment`, and `alcohol`. These fields reveal intimate personal habits that have no clear, documented relevance to creditworthiness assessment. Their collection is disproportionate to the stated processing purpose and likely constitutes a data minimisation violation.

**3. Quasi-Identifier Combination Risk**  
The simultaneous presence of `zip_code`, `gender`, `date_of_birth`, and `applicant_age` creates a strong re-identification vector. Research by Sweeney (2000) demonstrated that 87% of the US population can be uniquely identified using only ZIP code, gender, and date of birth — a combination present in full in this dataset.

These findings are addressed in the pseudonymisation demonstration (Section 3) and governance recommendations (Section 4).